# Clean and Normalize Product Dataset

Notebook này chỉ đọc dữ liệu trong `data/`, làm sạch, chuẩn hóa và xuất kết quả vào folder `processed/`.

Không tạo dữ liệu train, không train model.

In [ ]:
from pathlib import Path
import json
import re
import unicodedata

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
OUT_DIR = PROJECT_ROOT / "processed"
OUT_DIR.mkdir(exist_ok=True)

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 160)

## 1. Read All CSV Files

In [ ]:
csv_files = sorted(DATA_DIR.glob("*.csv"))
print(f"Found {len(csv_files)} CSV files")
for file in csv_files:
    print(file)

In [ ]:
def category_from_filename(path: Path) -> str:
    mapping = {
        "thong_tin_links_san_pham": "dien_thoai",
        "thong_tin_products_links": "dien_thoai",
        "thong_tin_links_laptop": "laptop",
        "thong_tin_links_pc": "pc",
        "thong_tin_links_screen": "man_hinh",
        "thong_tin_links_speaker": "loa",
        "thong_tin_links_tablet": "may_tinh_bang",
        "thong_tin_links_bluetooth": "tai_nghe_bluetooth",
        "thong_tin_links_micro": "micro",
        "thong_tin_links_accessory": "phu_kien",
    }
    return mapping.get(path.stem.lower(), path.stem.lower())

frames = []
for file in csv_files:
    df = pd.read_csv(file, encoding="utf-8-sig", dtype=str, keep_default_na=False)
    df["source_file"] = file.name
    df["category"] = category_from_filename(file)
    frames.append(df)

raw = pd.concat(frames, ignore_index=True, sort=False).fillna("")
print(raw.shape)
raw.head()

## 2. Clean Text and Column Names

In [ ]:
def clean_text(value) -> str:
    if pd.isna(value):
        return ""
    text = str(value).replace("\ufeff", "")
    text = text.replace("&amp;", "&")
    text = text.replace("&quot;", '"')
    text = text.replace("&gt;", ">")
    text = text.replace("&lt;", "<")
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"[\r\n\t]+", " ", text)
    text = re.sub(r"\s{2,}", " ", text)
    return text.strip()

def normalize_column_name(name: str) -> str:
    text = clean_text(name).lower()
    text = re.sub(r"\s+", "_", text)
    text = re.sub(r"_+", "_", text)
    return text.strip("_")

def make_unique_columns(columns: list) -> list:
    seen = {}
    unique = []
    for col in columns:
        count = seen.get(col, 0) + 1
        seen[col] = count
        unique.append(col if count == 1 else f"{col}_{count}")
    return unique

clean = raw.copy()
clean.columns = make_unique_columns([normalize_column_name(col) for col in clean.columns])

for col in clean.columns:
    clean[col] = clean[col].map(clean_text)

clean.head()

## 3. Standardize Core Fields

In [ ]:
def normalize_price(value: str):
    text = clean_text(value)
    if not text:
        return None
    lowered = text.lower()
    if lowered in {"liên hệ", "lien he", "đang cập nhật", "dang cap nhat"}:
        return None
    digits = re.sub(r"[^0-9]", "", text)
    return int(digits) if digits else None

required_cols = ["url", "ten_san_pham", "thuong_hieu", "gia_ban", "sku", "mo_ta_ngan", "category", "source_file"]
for col in required_cols:
    if col not in clean.columns:
        clean[col] = ""

before_rows = len(clean)
clean = clean[clean["ten_san_pham"].ne("")].copy()
clean["price_vnd"] = clean["gia_ban"].map(normalize_price)
clean["price_status"] = clean["price_vnd"].apply(lambda x: "available" if pd.notna(x) else "contact_or_missing")

clean = clean.drop_duplicates(subset=["url", "ten_san_pham", "sku"], keep="first").reset_index(drop=True)

print("Rows before:", before_rows)
print("Rows after removing empty product names and duplicates:", len(clean))

## 4. Build Structured Specs

In [ ]:
CORE_COLUMNS = {
    "url",
    "ten_san_pham",
    "thuong_hieu",
    "gia_ban",
    "price_vnd",
    "price_status",
    "sku",
    "mo_ta_ngan",
    "category",
    "source_file",
}

spec_columns = [col for col in clean.columns if col not in CORE_COLUMNS]

def row_specs(row) -> dict:
    specs = {}
    for col in spec_columns:
        value = row.get(col, "")
        if value:
            specs[col] = value
    return specs

def specs_to_text(specs: dict) -> str:
    return "; ".join(f"{key}: {value}" for key, value in specs.items())

clean["specs_json"] = clean.apply(lambda row: json.dumps(row_specs(row), ensure_ascii=False), axis=1)
clean["specs_text"] = clean["specs_json"].map(lambda value: specs_to_text(json.loads(value)))

products_clean = clean[[
    "url",
    "ten_san_pham",
    "thuong_hieu",
    "gia_ban",
    "price_vnd",
    "price_status",
    "sku",
    "mo_ta_ngan",
    "category",
    "source_file",
    "specs_text",
    "specs_json",
]].copy()

products_clean.head()

## 5. Create Long-Format Specs Table

In [ ]:
spec_rows = []
for _, row in clean.iterrows():
    specs = json.loads(row["specs_json"])
    for spec_name, spec_value in specs.items():
        spec_rows.append({
            "url": row["url"],
            "sku": row["sku"],
            "ten_san_pham": row["ten_san_pham"],
            "category": row["category"],
            "spec_name": spec_name,
            "spec_value": spec_value,
        })

specs_long = pd.DataFrame(spec_rows)
print(specs_long.shape)
specs_long.head()

## 6. Export to processed/

In [ ]:
products_clean.to_csv(OUT_DIR / "products_clean.csv", index=False, encoding="utf-8-sig")
products_clean.to_json(OUT_DIR / "products_clean.jsonl", orient="records", lines=True, force_ascii=False)

specs_long.to_csv(OUT_DIR / "product_specs_long.csv", index=False, encoding="utf-8-sig")
specs_long.to_json(OUT_DIR / "product_specs_long.jsonl", orient="records", lines=True, force_ascii=False)

category_counts = products_clean["category"].value_counts().rename_axis("category").reset_index(name="product_count")
category_counts.to_csv(OUT_DIR / "category_counts.csv", index=False, encoding="utf-8-sig")

report = {
    "input_files": [file.name for file in csv_files],
    "raw_rows": int(before_rows),
    "clean_rows": int(len(products_clean)),
    "removed_rows": int(before_rows - len(products_clean)),
    "spec_rows": int(len(specs_long)),
    "categories": category_counts.to_dict(orient="records"),
    "output_files": [
        "products_clean.csv",
        "products_clean.jsonl",
        "product_specs_long.csv",
        "product_specs_long.jsonl",
        "category_counts.csv",
        "processing_report.json",
    ],
}

with (OUT_DIR / "processing_report.json").open("w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print("Saved cleaned data to:", OUT_DIR.resolve())
category_counts